# Linear Algebra: Worked Solutions


In [ ]:
import numpy as np
from numpy.testing import assert_allclose
rng = np.random.default_rng(20260920)


## Problem 1. Apply an affine map to a row batch

**Problem.** Apply the column map $z=Ax+b$ to each of the two rows of $X$ below. Validate the input shapes and compare a batched calculation with separate applications.

**Solution.** Transposing the single-example equation gives $z^T=x^TA^T+b^T$, so $Z=XA^T+\mathbf1b^T$. Here the shapes are $X:(2,3)$, $A:(2,3)$, and $b:(2,)$. The resulting rows are $(4,-3)$ and $(6,-6)$. `b[None, :]` explicitly replicates the bias along the batch axis. A stored column of shape `(2,1)` is rejected rather than silently interpreted as a row bias.


In [ ]:
def affine_batch(X, A, b):
    X, A, b = map(lambda z: np.asarray(z, dtype=float), (X, A, b))
    if X.ndim != 2 or A.ndim != 2 or b.shape != (A.shape[0],) or X.shape[1] != A.shape[1]:
        raise ValueError("expected X:(B,d), A:(m,d), b:(m,)")
    return X @ A.T + b[None, :]


In [ ]:
X = np.array([[1., 2., -1.], [0., 4., 3.]])
A = np.array([[2., 0., 1.], [1., -1., 0.]])
b = np.array([3., -2.])
Z = affine_batch(X, A, b)
assert Z.shape == (2, 2)
assert_allclose(Z, np.stack([A @ x + b for x in X]))
assert np.ones(3).T.shape == (3,)
try:
    affine_batch(X, A, b[:, None])
except ValueError:
    pass
else:
    raise AssertionError("column storage must be converted explicitly")


## Problem 2. Project onto a line

**Problem.** Construct the orthogonal projector onto the line spanned by $u=(1,2)$ and project $b=(3,1)$. Check symmetry, idempotence, invariance under rescaling, and residual orthogonality.

**Solution.** The formula $P=uu^T/(u^Tu)$ gives $P=\frac15\begin{pmatrix}1&2\\2&4\end{pmatrix}$. Since $u^Tb=5$, the projection is $Pb=u=(1,2)$. The residual is $(2,-1)$, whose inner product with $u$ is zero. The factors introduced by replacing $u$ with $-7u$ cancel between numerator and denominator. A zero vector supplies no direction and must be rejected.


In [ ]:
def line_projection(u):
    u = np.asarray(u, dtype=float)
    if u.ndim != 1 or not np.any(u):
        raise ValueError("a nonzero one-dimensional coordinate array is required")
    return np.outer(u, u) / (u @ u)


In [ ]:
u = np.array([1., 2.]); b = np.array([3., 1.])
P = line_projection(u)
assert_allclose(P @ b, [1., 2.])
assert_allclose(P @ P, P)
assert_allclose(P.T, P)
assert_allclose(line_projection(-7*u), P)
assert_allclose(u @ (b-P@b), 0., atol=1e-14)
try:
    line_projection(np.zeros(2))
except ValueError:
    pass
else:
    raise AssertionError("zero does not define a line")


## Problem 3. Solve least squares with QR

**Problem.** Minimize $\|Ax-b\|_2$ for $A=\begin{pmatrix}1&0\\1&1\\1&2\end{pmatrix}$ and $b=(1,2,2)^T$. Implement a QR solution for full-column-rank matrices and check the residual.

**Solution.** If $A=QR$ with orthonormal columns in $Q$, the minimizer solves $Rx=Q^Tb$. For this example the equivalent equations are $3x_1+3x_2=5$ and $3x_1+5x_2=6$, yielding $x=(7/6,1/2)$. The residual $b-Ax=(-1/6,1/3,-1/6)$ is orthogonal to both columns of $A$. The implementation solves the triangular system without forming an inverse. The additional random example compares against `lstsq` and checks the same orthogonality condition.


In [ ]:
def qr_least_squares(A, b):
    Q, R = np.linalg.qr(np.asarray(A, dtype=float), mode="reduced")
    return np.linalg.solve(R, Q.T @ np.asarray(b, dtype=float))


In [ ]:
A = np.array([[1., 0.], [1., 1.], [1., 2.]])
b = np.array([1., 2., 2.])
x = qr_least_squares(A,b)
assert_allclose(x, [7/6, 1/2])
assert_allclose(A.T @ (b-A@x), 0., atol=1e-14)
Q, _ = np.linalg.qr(rng.normal(size=(7,3)), mode="reduced")
A2 = Q @ np.diag([3.,2.,1.]); b2 = rng.normal(size=7)
x2 = qr_least_squares(A2,b2)
assert_allclose(x2, np.linalg.lstsq(A2,b2,rcond=None)[0])
assert_allclose(A2.T @ (b2-A2@x2), 0., atol=1e-13)


## Problem 4. Select the minimum-norm solution

**Problem.** For $A=\begin{pmatrix}1&1\\2&2\end{pmatrix}$ and $b=(1,2)^T$, recover the minimum-norm least-squares solution using SVD. Then examine what changes when a numerical cutoff discards a small, nonzero singular value.

**Solution.** All exact solutions satisfy $x_1+x_2=1$. Writing them as $(1/2,1/2)+t(1,-1)$ shows that their squared norms are $1/2+2t^2$; the minimum occurs at $t=0$. In $A=U\operatorname{diag}(s)V^T$, the pseudoinverse multiplies $U^Tb$ by reciprocal nonzero singular values and maps back through $V$.

The implementation retains values larger than `rtol*s[0]`. For the exactly zero matrix it returns the zero vector, which is the minimum-norm minimizer regardless of $b$. For $\operatorname{diag}(1,10^{-10})$, cutoff $10^{-8}$ instead discards a genuinely nonzero direction and returns $(1,0)$; cutoff $10^{-12}$ retains it and returns $(1,10^{10})$. The former solves the *truncated* matrix problem and is not an exact least-squares minimizer for the original invertible matrix. A numerical rank convention changes which inverse directions are retained.


In [ ]:
def minimum_norm(A, b, rtol=1e-12):
    A, b = np.asarray(A, dtype=float), np.asarray(b, dtype=float)
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    keep = s > rtol * s[0]
    if not np.any(keep):
        return np.zeros(A.shape[1])
    return Vt[keep].T @ ((U[:,keep].T @ b) / s[keep])


In [ ]:
A = np.array([[1., 1.], [2., 2.]])
b = np.array([1., 2.])
x = minimum_norm(A,b)
assert_allclose(x, [.5,.5])
assert_allclose(A @ x,b)
for t in [-3.,-.2,1.,4.]:
    alternative = x + t*np.array([1.,-1.])
    assert_allclose(A@alternative,b)
    assert np.linalg.norm(alternative) > np.linalg.norm(x)
assert_allclose(minimum_norm(np.zeros((2,3)),[2,3]), np.zeros(3))
small = np.diag([1.,1e-10])
assert_allclose(minimum_norm(small, [1.,1.],rtol=1e-8), [1.,0.])
assert_allclose(minimum_norm(small, [1.,1.],rtol=1e-12), [1.,1e10])


## Problem 5. Truncate an SVD

**Problem.** Construct a matrix with singular values $5,2,1$, compute its best rank-one approximation, and determine its Frobenius and operator errors. Also check rank-zero and full-rank reconstruction.

**Solution.** Keeping only the first singular term discards singular values two and one. Thus the squared Frobenius error is $2^2+1^2=5$, and the operator error is two. The reconstruction has rank one. Retaining zero terms gives the zero map; retaining all three reconstructs the original matrix. The checks compare these invariant quantities, since changing signs of paired singular vectors does not change the represented matrix.


In [ ]:
def rank_k_approximation(A, k):
    A = np.asarray(A, dtype=float)
    if not 0 <= k <= min(A.shape):
        raise ValueError("rank bound is outside the allowed range")
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    return (U[:,:k] * s[:k]) @ Vt[:k]


In [ ]:
U,_ = np.linalg.qr(rng.normal(size=(5,3)), mode="reduced")
V,_ = np.linalg.qr(rng.normal(size=(3,3)))
A = U @ np.diag([5.,2.,1.]) @ V.T
Ak = rank_k_approximation(A,1)
assert_allclose(np.linalg.norm(A-Ak,"fro")**2, 5., atol=1e-12)
assert_allclose(np.linalg.norm(A-Ak,2), 2., atol=1e-12)
assert np.linalg.matrix_rank(Ak,tol=1e-10) == 1
assert_allclose(rank_k_approximation(A,0),np.zeros_like(A))
assert_allclose(rank_k_approximation(A,3),A,atol=1e-13)


## Problem 6. Measure directional sensitivity

**Problem.** For $A=\operatorname{diag}(1,10^{-6})$ and $b=(1,0)$, compare changes in the solution under equal-length perturbations $(10^{-6},0)$ and $(0,10^{-6})$. Compare the condition numbers of $A$ and $A^TA$.

**Solution.** The original solution is $(1,0)$. The first perturbation changes it by $(10^{-6},0)$; the second changes it by $(0,1)$. Their amplification factors are respectively one and $10^6$. The small singular-value direction is sensitive because the solve divides by $10^{-6}$. Therefore $\kappa_2(A)=10^6$, whereas $A^TA=\operatorname{diag}(1,10^{-12})$ has condition number $10^{12}=\kappa_2(A)^2$. These are properties of the linear maps, not a dependence on a particular rounding-error outcome.


In [ ]:
A = np.diag([1.,1e-6])
b = np.array([1.,0.]); x = np.linalg.solve(A,b)
perturb_good = np.array([1e-6,0.]); perturb_bad = np.array([0.,1e-6])
dx_good = np.linalg.solve(A,b+perturb_good)-x
dx_bad = np.linalg.solve(A,b+perturb_bad)-x
assert_allclose(np.linalg.norm(dx_good),1e-6)
assert_allclose(np.linalg.norm(dx_bad),1.)
assert_allclose(np.linalg.cond(A.T@A),np.linalg.cond(A)**2)
print("Sensitivity amplification:", np.linalg.norm(dx_bad)/np.linalg.norm(perturb_bad))
